# TP2 - Modeling using docplex

## 1. The `docplex` python package

`DOcplex` is a python package developped by IBM &mdash; It provides easy-to-use API for IBM solvers Cplex and Cpoptimizer.

DOcplex documentation for mathematical programming can be found here: http://ibmdecisionoptimization.github.io/docplex-doc/#mathematical-programming-modeling-for-python-using-docplex-mp-docplex-mp

## 2. Solving TSP using `docplex`

### 2.1. TSP model using `docplex`

**Exercice:** Using `docplex`, create a model for the travelling salesman problem using the MTZ or Flow formulation and compare them.

In [1]:
from docplex.mp.model import Model
import tsp.data as data

distances = data.grid26
N = len(distances)

tsp = Model("TSP")
tsp.log_output = True

# nos variables : on a les arcs empruntés et le rang des villes dans le parcours
x = tsp.binary_var_matrix(N, N, name="x")
u = tsp.continuous_var_dict(range(1,N), lb=1, ub=N ,name="u")

# on interdit l'algo de boucler sur une meme ville
for i in range(N):
    tsp.add_constraint(x[i,i] == 0)
# il faut absolument qu'on passe et qu'on sorte une fois dans chaque ville exactement
for i in range(N):
    tsp.add_constraint(tsp.sum(x[i,j] for j in range(N) if j != i)==1)
    tsp.add_constraint(tsp.sum(x[j,i] for j in range(N) if j != i)==1)
# contraintes MTZ pour empecher les sous tours 
for i in range(1, N):
    for j in range(1, N):
        if i != j:
            tsp.add_constraint(u[i] - u[j] + N * x[i, j] <= N - 1)
# écriture du problème d'optimisation
tsp.minimize(tsp.sum(distances[i][j] * x[i,j] for i in range(N) for j in range(N)))

solution = tsp.solve()
print("z* =", solution.objective_value)


Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
MIP Presolve eliminated 26 rows and 26 columns.
Reduced MIP has 652 rows, 675 columns, and 3100 nonzeros.
Reduced MIP has 650 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (1.84 ticks)
Found incumbent of value 2591.000000 after 0.03 sec. (6.18 ticks)
Probing time = 0.00 sec. (2.75 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 652 rows, 675 columns, and 3100 nonzeros.
Reduced MIP has 650 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (2.41 ticks)
Probing time = 0.00 sec. (2.74 ticks)
Clique table members: 352.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 12 threads.
Root relaxation solution time = 0.00 sec. (1.01 ticks)

        Nodes                                         Cuts/
   Node  Left     O

In [2]:
from docplex.mp.model import Model
import tsp.data as data

distances = data.grid42
N = len(distances)

tsp = Model("TSP")
tsp.log_output = True

# x[i, j] : arc emprunté ou non
x = tsp.binary_var_matrix(N, N, name="x")

# y[i, j] : quantité de flot transportée sur cet arc
y = tsp.continuous_var_matrix(N, N, lb=0, name="y")

# Interdire les boucles
for i in range(N):
    tsp.add_constraint(x[i, i] == 0)

# Exactement un départ et une arrivée par ville
for i in range(N):
    tsp.add_constraint(
        tsp.sum(x[i, j] for j in range(N) if j != i) == 1
    )
    tsp.add_constraint(
        tsp.sum(x[j, i] for j in range(N) if j != i) == 1
    )

# La ville 0 fournit N - 1 unités de flot
tsp.add_constraint(
    tsp.sum(y[0, j] for j in range(1, N))
    - tsp.sum(y[j, 0] for j in range(1, N))
    == N - 1
)

# Chaque autre ville consomme une unité
for i in range(1, N):
    tsp.add_constraint(
        tsp.sum(y[j, i] for j in range(N))
        - tsp.sum(y[i, j] for j in range(N))
        == 1
    )

# Le flot ne circule que sur les arcs empruntés
for i in range(N):
    for j in range(N):
        tsp.add_constraint(y[i, j] <= (N - 1) * x[i, j])

# Minimiser la distance totale du circuit
tsp.minimize(
    tsp.sum(
        distances[i][j] * x[i, j]
        for i in range(N)
        for j in range(N)
    )
)

solution = tsp.solve()
print("z* =", solution.objective_value)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
MIP Presolve eliminated 84 rows and 84 columns.
Reduced MIP has 1848 rows, 3444 columns, and 10332 nonzeros.
Reduced MIP has 1722 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (5.32 ticks)
Found incumbent of value 3539.000000 after 0.02 sec. (11.06 ticks)
Probing time = 0.03 sec. (8.04 ticks)
Cover probing fixed 0 vars, tightened 164 bounds.
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 83 rows and 82 columns.
MIP Presolve modified 165 coefficients.
Reduced MIP has 1765 rows, 3362 columns, and 10045 nonzeros.
Reduced MIP has 1722 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (9.67 ticks)
Probing time = 0.02 sec. (3.89 ticks)
Cover probing fixed 0 vars, tightened 360 bounds.
Clique table members: 84.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic s

The largest set of distances contains 42 nodes, and should be easily solved by `docplex`.

### 2.2. Generating random TSP instances

**Question:** What method could you implement to generate a realistic set of distances for $n$ customers?

**Exercice:** Implement the method proposed above and test it.

In [3]:
import numpy as np
import scipy.spatial.distance


def generate_distances(n: int):
    points = np.random.uniform(0,100, size=(n,2))
    return scipy.spatial.distance.cdist(points,points,metric="euclidean")


from docplex.mp.model import Model

distances = generate_distances(42)
print(distances)

N = len(distances)

tsp = Model("TSP")
tsp.log_output = True

# nos variables : on a les arcs empruntés et le rang des villes dans le parcours
x = tsp.binary_var_matrix(N, N, name="x")
u = tsp.continuous_var_dict(range(1,N), lb=1, ub=N ,name="u")

# on interdit l'algo de boucler sur une meme ville
for i in range(N):
    tsp.add_constraint(x[i,i] == 0)
# il faut absolument qu'on passe et qu'on sorte une fois dans chaque ville exactement
for i in range(N):
    tsp.add_constraint(tsp.sum(x[i,j] for j in range(N) if j != i)==1)
    tsp.add_constraint(tsp.sum(x[j,i] for j in range(N) if j != i)==1)
# contraintes MTZ pour empecher les sous tours 
for i in range(1, N):
    for j in range(1, N):
        if i != j:
            tsp.add_constraint(u[i] - u[j] + N * x[i, j] <= N - 1)
# écriture du problème d'optimisation
tsp.minimize(tsp.sum(distances[i][j] * x[i,j] for i in range(N) for j in range(N)))

solution = tsp.solve()
print("z* =", solution.objective_value)


[[ 0.         23.34938231 48.1968409  ... 54.77614061 25.18802601
  48.2490366 ]
 [23.34938231  0.         49.19940122 ... 60.89415083 48.28465727
  52.99527011]
 [48.1968409  49.19940122  0.         ... 15.02877385 62.55370531
   8.25913375]
 ...
 [54.77614061 60.89415083 15.02877385 ...  0.         63.41962055
   8.06754897]
 [25.18802601 48.28465727 62.55370531 ... 63.41962055  0.
  59.36100226]
 [48.2490366  52.99527011  8.25913375 ...  8.06754897 59.36100226
   0.        ]]
Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
MIP Presolve eliminated 42 rows and 42 columns.
Reduced MIP has 1724 rows, 1763 columns, and 8364 nonzeros.
Reduced MIP has 1722 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (4.30 ticks)
Found incumbent of value 1848.119815 after 0.02 sec. (11.67 ticks)
Probing time = 0.02 sec. (3.34 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 

## 3. Solving Warehouse Allocation using Benders decomposition with `docplex`

### 3.1. The warehouse problem

A company needs to supply a set of $n$ clients and needs to open new warehouses (from a
set of $m$ possible warehouses).
Opening a warehouse $j$ costs $f_j$ and supplying a client $i$ from a warehouse $j$ costs $c_{ij}$ per supply unit.
Which warehouses should be opened in order to satisfy all clients while minimizing the total cost?

### 3.2. Solving the warehouse problem with a single ILP

- $y_{j} = 1$ if and only if warehouse $j$ is opened.
- $x_{ij}$ is the fraction supplied from warehouse $j$ to customer $i$.

$
\begin{align}
  \text{min.} \quad & \sum_{i=1}^{n} \sum_{j=1}^{m} c_{ij} x_{ij} + \sum_{j=1}^{m} f_{j} y_{j} & \\
  \text{s.t.} \quad & \sum_{j=1}^{m} x_{ij} = 1, & \forall i \in\{1,\ldots,n\}\\
                    & x_{ij} \leq y_{j}, & \forall i\in\{1,\ldots,n\},\forall j\in\{1,\ldots,m\}\\
                    & y_{j} \in \left\{0,~1\right\}, & \forall j \in \left\{1,~\ldots,~m\right\}\\
                    & x_{ij} \geq 0, & \forall i \in \left\{1,~\ldots,~n\right\}, \forall j \in \left\{1,~\ldots,~m\right\}
\end{align}
$


**Exercice:** Implement the ILP model for the warehouse allocation problem and test it on the given instance.

In [4]:
from docplex.mp.model import Model

# We will start with a small instances with 3 warehouses and 3 clients:
N = 3
M = 3

# Opening and distribution costs:
f = [20, 20, 20]
c = [[15, 1, 2], [1, 16, 3], [4, 1, 17]]

wa = Model("Warehouse Allocation")
wa.log_output = True

# y[j] = 1 if warehouse j ouvert sinon 0
y = wa.binary_var_list(M, name="y")

# x[i,j] : fraction du client i fournie par l'entrepot j
x =wa.continuous_var_matrix(N,M,lb=0,name="x")

#Chaque client reçoit 100% de sa demande
for i in range(N):
    wa.add_constraint(wa.sum(x[i,j] for j in range(M)) == 1)

# Si un entrepot est fermé, alors aucun client ne peut être servi par lui
for j in range(M):
    for i in range(N):
        wa.add_constraint(x[i,j] <= y[j])

# Minimize the total cost
wa.minimize(wa.sum(f[j] * y[j] for j in range(M)) + wa.sum(c[i][j] * x[i,j] for i in range(N) for j in range(M)))

solution = wa.solve()
print("z* =", solution.objective_value)


Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
Reduced MIP has 12 rows, 12 columns, and 27 nonzeros.
Reduced MIP has 3 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
Found incumbent of value 80.000000 after 0.00 sec. (0.03 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 12 rows, 12 columns, and 27 nonzeros.
Reduced MIP has 3 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.02 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 12 threads.
Root relaxation solution time = 0.02 sec. (0.02 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0               

### 3.3. Benders' decomposition for the Warehouse Allocation problem

We are going to use Benders' decomposition to solve the Warehouse Allocation problem. 

#### Dual subproblem

$
\begin{align*}
\text{max.} \quad & \sum_{i=1}^{n} v_{i} - \sum_{i=1}^{n}\sum_{j=1}^{m} \bar{y}_{j} u_{ij} & \\
\text{s.t.} \quad & v_{i} - u_{ij} \leq c_{ij}, & \forall i\in\{1,\ldots,n\},\forall j\in\{1,\ldots,m\}\\
                  & v_{i} \in\mathbb{R},\ u_{ij} \geq 0 & \forall i \in\{1,\ldots,n\}, \forall j\in\{1,\ldots,m\}
\end{align*}
$

#### Master problem

$
\begin{align*}
  \text{min.} \quad & \sum_{j=1}^{m} f_j y_j + z & \\
  \text{s.t.} \quad & z \geq \sum_{i=1}^{n}v_i^p - \sum_{i=1}^{n} \sum_{j=1}^{m} u_{ij}^p y_j, & \forall p\in l_1\\
                    & 0 \geq \sum_{i=1}^{n}v_i^r - \sum_{i=1}^{n} \sum_{j=1}^{n} u_{ij}^r y_j, & \forall r\in l_2\\
                    & y_{j} \in\{0,1\}, & \forall j\in\{1,\ldots,m\}
\end{align*}
$

**Exercice:** Implement the method `create_master_problem` that creates the initial master problem (without feasibility or optimality constraints) for the warehouse allocation problem.

<div class="alert alert-info alert-block">

You can use `print(m.export_as_lp_string())` to display a textual representation of a `docplex` model `m`.
    
</div>

In [5]:
from docplex.mp.model import Model
from docplex.mp.linear import Var
from docplex.mp.constr import AbstractConstraint
from typing import List, Sequence, Tuple


def create_master_problem(
    N: int, M: int, f: Sequence[float], c: Sequence[Sequence[float]]
) -> Tuple[Model, Var, Sequence[Var]]:
    """
    Creates the initial Benders master problem for the Warehouse Allocation problem.

    Args:
        N: Number of clients.
        M: Number of warehouses.
        f: Array-like containing costs of opening warehouses.
        c: 2D-array like containing transport costs from client to warehouses.

    Returns:
        A 3-tuple containing the docplex problem, the z variable and the y variables.
    """

    wa = Model("Warehouse Allocation - Benders master problem")

    y = wa.binary_var_list(M, name="y")  #decision d'ouverture
    z = wa.continuous_var(lb=0, name="z")  #variable de coût de livraison

    #cout d'ouverture + estimation du coup de livraison 
    wa.minimize(wa.sum(f[j] * y[j] for j in range(M)) + z)

    return wa, z, y


# Check your method:
wa, z, y = create_master_problem(N, M, f, c)
print(wa.export_as_lp_string())


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: Warehouse Allocation - Benders master problem

Minimize
 obj: 20 y_0 + 20 y_1 + 20 y_2 + z
Subject To

Bounds
 0 <= y_0 <= 1
 0 <= y_1 <= 1
 0 <= y_2 <= 1

Binaries
 y_0 y_1 y_2
End



**Exercice:** Implement the method `add_optimality_constraints` that add optimality constraints to the Benders master problem. 

In [6]:
def add_optimality_constraint(
    N: int,
    M: int,
    wa: Model,
    z: Var,
    y: Sequence[Var],
    v: Sequence[float],
    u: Sequence[Sequence[float]],
) -> List[AbstractConstraint]:
    """
    Adds an optimality constraints to the given Warehouse Allocation model
    using the given optimal values from the Benders dual subproblem.

    Args:
        N: Number of clients.
        M: Number of warehouses.
        wa: The Benders master problem (docplex.mp.model.Model).
        z: The z variable of the master problem.
        y: The y variables of the master problem.
        v: The optimal values for the v variables of the Benders dual subproblem.
        u: The optimal values for the u variables of the Benders dual subproblem.

    Return: The optimality constraint added.
    """
    return wa.add_constraint( z>= sum(v[i] for i in range(N))-wa.sum(u[i][j]*y[j] for i in range(N) for j in range(M)))


**Exercice:** Implement the method `add_feasibility_constraints` that add feasibility constraints to the Benders master problem. 

In [7]:
def add_feasibility_constraints(
    N: int,
    M: int,
    wa: Model,
    z: Var,
    y: Sequence[Var],
    v: Sequence[float],
    u: Sequence[Sequence[float]],
) -> List[AbstractConstraint]:
    """
    Adds an optimality constraints to the given Warehouse Allocation model
    using the given optimal values from the Benders dual subproblem.

    Args:
      - N: Number of clients.
      - M: Number of warehouses.
      - wa: The Benders master problem (docplex.mp.model.Model).
      - z: The z variable of the master problem.
      - y: The y variables of the master problem.
      - v: The extreme rays for the v variables of the Benders dual subproblem.
      - u: The extreme rays for the u variables of the Benders dual subproblem.

    Returns:
        The feasibility constraint added.
    """
    return wa.add_constraint(
        sum(v[i] for i in range(N))
        - wa.sum(
            u[i][j] * y[j]
            for i in range(N)
            for j in range(M)
        )
        <= 0
    )


**Exercice:** Implement the method `create_dual_subproblem` that, given a solution `y` of the master problem, create the corresponding Benders dual subproblem.

$
\begin{align*}
\text{max.} \quad & \sum_{i=1}^{n} v_{i} - \sum_{i=1}^{n}\sum_{j=1}^{m} \bar{y}_{j} u_{ij} & \\
\text{s.t.} \quad & v_{i} - u_{ij} \leq c_{ij}, & \forall i\in\{1,\ldots,n\},\forall j\in\{1,\ldots,m\}\\
                  & v_{i} \in\mathbb{R},\ u_{ij} \geq 0 & \forall i \in\{1,\ldots,n\}, \forall j\in\{1,\ldots,m\}
\end{align*}
$

In [8]:
from docplex.mp.model import Model


def create_dual_subproblem(
    N: int, M: int, f: Sequence[float], c: Sequence[Sequence[float]], y: Sequence[int]
) -> Tuple[Model, Sequence[Var], Sequence[Sequence[Var]]]:
    """
    Creates a Benders dual subproblem for the Warehouse Allocation problem corresponding
    to the given master solution.

    Args:
        N: Number of clients.
        M: Number of warehouses.
        f: Array-like containing costs of opening warehouses.
        c: 2D-array like containing transport costs from client to warehouses.
        y: Values of the y variables from the Benders master problem.

    Returns:
        A 3-tuple containing the docplex problem, the v variable and the u variables.
    """

    dsp = Model("Warehouse Allocation - Benders dual subproblem")

    # We disable pre-solve to be able to retrieve a meaningful status in the main
    # algorithm:
    dsp.parameters.preprocessing.presolve.set(0)

    # v[i] est libre : peut être négatif ou positif
    v = dsp.continuous_var_list( N, lb=-dsp.infinity, name="v")

    # u[i][j] est positif ou nul
    u = [dsp.continuous_var_list(M, lb=0, name=f"u_{i}") for i in range(N)]

    # Contraintes du dual
    for i in range(N):
        for j in range(M):
            dsp.add_constraint(v[i] - u[i][j] <= c[i][j])

    # Objectif du dual pour les ouvertures y fixées
    dsp.maximize( dsp.sum(v[i] for i in range(N)) - dsp.sum( y[j] * u[i][j] for i in range(N) for j in range(M)))

    return dsp, v, u


# Check your method (assuming y = [1 1 1 ... 1]):
dsp, v, u = create_dual_subproblem(N, M, f, c, [1] * M)
print(dsp.export_as_lp_string())


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: Warehouse Allocation - Benders dual subproblem

Maximize
 obj: v_0 + v_1 + v_2 - u_0_0 - u_0_1 - u_0_2 - u_1_0 - u_1_1 - u_1_2 - u_2_0
      - u_2_1 - u_2_2
Subject To
 c1: v_0 - u_0_0 <= 15
 c2: v_0 - u_0_1 <= 1
 c3: v_0 - u_0_2 <= 2
 c4: v_1 - u_1_0 <= 1
 c5: v_1 - u_1_1 <= 16
 c6: v_1 - u_1_2 <= 3
 c7: v_2 - u_2_0 <= 4
 c8: v_2 - u_2_1 <= 1
 c9: v_2 - u_2_2 <= 17

Bounds
       v_0 Free
       v_1 Free
       v_2 Free
End



**Exercice:** Using the methods you implemented, write the Benders decomposition algorithm for the warehouse allocation problem.

<div class="alert alert-block alert-info">

The `get_extreme_rays` function can be used to retrieve the extreme rays associated with an unbounded solution of the dual subproblem.
    
</div>

<div class="alert alert-block alert-info">
    
You can use `model.get_solve_status()` to obtain the status of the resolution and compare it to members of `JobSolveStatus`:
    
```python
if model.get_solve_status() == JobSolveStatus.OPTIMAL_SOLUTION:
    pass
```
    
</div>

In [9]:
from docplex.mp.model import Model
from docplex.util.status import JobSolveStatus


def get_extreme_rays(
    N: int, M: int, model: Model, v: Sequence[Var], u: Sequence[Sequence[Var]]
) -> Tuple[Sequence[float], Sequence[Sequence[float]]]:
    """
    Retrieves the extreme rays associated to the dual subproblem.

    Args:
        N: Number of clients.
        M: Number of warehouses.
        model: The Benders dual subproblem model (docplex.mp.model.Model).
        v: 1D array containing the v variables of the subproblem.
        u: Either a 2D array of a tuple-index dictionary containing the u variables
            of the subproblem.

    Returns:
        A 2-tuple containing the list of extreme rays correspondig to v,
        and the 2D-list of extreme rays corresponding to u.
    """
    ray = model.get_engine().get_cplex().solution.advanced.get_ray()

    if isinstance(u, dict):

        def get_uij(i, j):
            return u[i, j]

    else:

        def get_uij(i, j):
            return u[i][j]

    return (
        [ray[v[i].index] for i in range(N)],
        [[ray[get_uij(i, j).index] for j in range(M)] for i in range(N)],
    )


# We will start with a small instances with 3 warehouses and 3 clients:
N = 3
M = 3

# Opening and distribution costs:
f = [20, 20, 20]
c = [[15, 1, 2], [1, 16, 3], [4, 1, 17]]

# We stop iterating if the new solution is less than epsilon
# better than the previous one:
epsilon = 1e-6

wa, z, y = create_master_problem(N, M, f, c)

n = 0
while True:

    # Print iteration:
    n = n + 1
    print("Iteration {}".format(n))

    # 1. Résoudre le maître et récupérer les ouvertures proposées
    master_solution = wa.solve()

    if wa.get_solve_status() != JobSolveStatus.OPTIMAL_SOLUTION:
        raise RuntimeError(f"Maître non optimal : {wa.solve_details.status}")

    y_values = [round(master_solution.get_value(y[j])) for j in range(M)]
    z_value = master_solution.get_value(z)

    print("Ouvertures :", y_values, "| Coût estimé des livraisons :", z_value)

    # 2. Construire et résoudre le dual pour ces ouvertures
    dsp, v, u = create_dual_subproblem(N, M, f, c, y_values)
    dsp.solve()

    status = dsp.get_solve_status()

    if status == JobSolveStatus.OPTIMAL_SOLUTION:
        delivery_cost = dsp.objective_value

        # L'estimation du maître est suffisamment précise : arrêt
        if delivery_cost <= z_value + epsilon:
            print("Coût optimal :", master_solution.objective_value)
            print("Entrepôts ouverts :", y_values)
            break

        # Sinon, améliorer l'estimation avec une coupe d'optimalité
        v_values = [dsp.solution.get_value(v[i]) for i in range(N)]
        u_values = [
            [dsp.solution.get_value(u[i][j]) for j in range(M)]
            for i in range(N)
        ]

        add_optimality_constraint(N, M, wa, z, y, v_values, u_values)
        print("Ajout d'une coupe d'optimalité")

    elif status == JobSolveStatus.UNBOUNDED_SOLUTION:
        # Ces ouvertures ne permettent pas de livrer tous les clients
        v_ray, u_ray = get_extreme_rays(N, M, dsp, v, u)

        add_feasibility_constraints(N, M, wa, z, y, v_ray, u_ray)
        print("Ajout d'une coupe de faisabilité")

    else:
        raise RuntimeError(f"Statut inattendu du dual : {dsp.solve_details.status}")

    

print("Done.")


Iteration 1
Ouvertures : [0, 0, 0] | Coût estimé des livraisons : 0
Ajout d'une coupe de faisabilité
Iteration 2
Ouvertures : [1, 0, 0] | Coût estimé des livraisons : 0
Ajout d'une coupe d'optimalité
Iteration 3
Ouvertures : [0, 1, 0] | Coût estimé des livraisons : 3.0
Ajout d'une coupe d'optimalité
Iteration 4
Ouvertures : [0, 0, 1] | Coût estimé des livraisons : 7.0
Ajout d'une coupe d'optimalité
Iteration 5
Ouvertures : [0, 1, 0] | Coût estimé des livraisons : 18.0
Coût optimal : 38.0
Entrepôts ouverts : [0, 1, 0]
Done.


<div style="color: #1565C0;">

L’algorithme de Benders converge en **5 itérations**. La première ajoute une coupe de faisabilité pour interdire de fermer tous les entrepôts. Les trois suivantes ajoutent des coupes d’optimalité pour affiner l’estimation du coût de livraison.

À la dernière itération, le coût estimé rejoint le coût réel de livraison : **18**. La solution optimale consiste à ouvrir uniquement le **deuxième entrepôt**, soit **y = [0, 1, 0]**.

Le coût total est de **20 + 18 = 38**, ce qui confirme le résultat obtenu avec le modèle complet en partie 3.2.

</div>

### 3.4. Generating instances for the Warehouse Allocation problem

**Exercice:** Using the TSP instances contained in `tsp.data` or the `generate_distances` method, create instances for the warehouse allocation problem with randomized opening costs.

In [10]:
import numpy as np

# Fixer la graine pour pouvoir reproduire la même instance
np.random.seed(42)

N = 10  # Nombre de clients
M = 5   # Nombre d'entrepôts possibles

# Générer les distances entre N + M positions :
# les N premières sont les clients, les M suivantes les entrepôts
distances = generate_distances(N + M)

# Coûts de livraison : distance du client i à l'entrepôt j
c = distances[:N, N:].tolist()

# Coûts d'ouverture aléatoires entre 20 et 100 inclus
f = np.random.randint(20, 101, size=M).tolist()

print("Nombre de clients :", N)
print("Nombre d'entrepôts :", M)
print("Coûts d'ouverture :", f)
print("Coûts de livraison :")
print(np.round(c, 2))

Nombre de clients : 10
Nombre d'entrepôts : 5
Coûts d'ouverture : [70, 26, 40, 92, 58]
Coûts de livraison :
[[ 84.52  59.01  18.45  47.02  93.01]
 [ 47.46  49.74  33.31  53.9   56.96]
 [ 45.61  25.06  69.71  36.09  44.99]
 [ 91.36  55.19  40.61  37.94  97.85]
 [ 56.87  46.07  16.43  44.58  66.17]
 [101.94  66.18  47.3   48.96 108.62]
 [ 23.23  56.18  68.54  70.11  29.18]
 [ 43.23  21.36  66.13  33.13  43.28]
 [ 49.3   15.89  30.14  10.51  55.84]
 [ 23.54  15.87  49.45  32.2   29.27]]


<div class="alert alert-block alert-danger"></div>